# Graceful Degradation: Practice Exercise

Build a stock information agent that gracefully degrades across three data sources when failures occur. You will implement the routing logic and state transitions for multi-level fallback.

**What you'll implement:**
- Workflow nodes that update state based on success/failure
- Two routing functions for conditional degradation
- Graph assembly with conditional edges

**Estimated time:** 15 minutes

## Setup

Run this cell to import all required libraries and initialize the API clients.

In [ ]:
# Setup - run this cell first

import os
import logging
from datetime import datetime
from typing import TypedDict, Optional, List, Literal

from dotenv import load_dotenv
from tavily import TavilyClient

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

# Load environment variables
load_dotenv()

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('stock_workflow')

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

print("Setup complete!")

## Scenario

You are building a stock information agent that provides company stock data. The agent has three data sources with decreasing quality:

| Level | Source | Quality | Description |
|-------|--------|---------|-------------|
| 0 | Mock Stock API | High | Simulated real-time stock data |
| 1 | Web Search | Medium | Recent stock news from web search |
| 2 | LLM Knowledge | Low | General company information from LLM |

**Workflow behavior:**
- Start with Level 0 (Mock API)
- If Level 0 fails, automatically fall back to Level 1 (Web Search)
- If Level 1 fails, automatically fall back to Level 2 (LLM Knowledge)
- Track errors and data quality throughout

## Provided: State Schema and Data Source Functions

The state schema and data source functions are provided. Each data source returns a tuple of `(success: bool, result: str)`.

In [ ]:
# State schema - provided

class StockWorkflowState(TypedDict):
    """State for stock workflow with graceful degradation."""
    ticker: str                        # Company ticker symbol
    degradation_level: int             # 0=API, 1=Search, 2=LLM
    result: Optional[str]              # Stock information
    error_log: List[str]               # Track failures at each level
    data_quality: str                  # "high", "medium", "low"


print("State schema defined!")

In [ ]:
# Data source functions - provided
# These simulate different data sources with varying reliability

# Simulated stock data (acts as our "API")
MOCK_STOCK_DATA = {
    "AAPL": {"price": 178.52, "change": 2.34, "volume": "52.3M"},
    "GOOGL": {"price": 141.80, "change": -1.25, "volume": "18.7M"},
    "MSFT": {"price": 378.91, "change": 4.12, "volume": "22.1M"},
}

# Flags to simulate failures for testing
SIMULATE_API_FAILURE = False
SIMULATE_SEARCH_FAILURE = False


def fetch_from_stock_api(ticker: str) -> tuple[bool, str]:
    """Level 0: Fetch stock data from mock API."""
    logger.info(f"[Level 0] Attempting Stock API for {ticker}")
    
    if SIMULATE_API_FAILURE:
        return (False, "API service unavailable")
    
    if ticker not in MOCK_STOCK_DATA:
        return (False, f"Ticker {ticker} not found in database")
    
    data = MOCK_STOCK_DATA[ticker]
    result = f"""Stock Data for {ticker}:
- Current Price: ${data['price']}
- Change: ${data['change']:+.2f}
- Volume: {data['volume']}
- Source: Real-time API data"""
    
    return (True, result)


def fetch_from_web_search(ticker: str) -> tuple[bool, str]:
    """Level 1: Fetch stock information from web search."""
    logger.info(f"[Level 1] Attempting web search for {ticker}")
    
    if SIMULATE_SEARCH_FAILURE:
        return (False, "Web search service unavailable")
    
    try:
        tavily = TavilyClient()
        response = tavily.search(
            query=f"{ticker} stock price news today",
            max_results=3,
            search_depth="basic"
        )
        
        if not response.get("results"):
            return (False, "No search results found")
        
        summaries = [r.get("content", "")[:200] for r in response["results"][:3]]
        result = f"""Web Search Results for {ticker}:

{chr(10).join(f'- {s}...' for s in summaries)}

Source: Web search (may not reflect current prices)"""
        
        return (True, result)
        
    except Exception as e:
        return (False, f"Search error: {str(e)}")


def fetch_from_llm_knowledge(ticker: str) -> tuple[bool, str]:
    """Level 2: Get general company information from LLM knowledge."""
    logger.info(f"[Level 2] Using LLM knowledge for {ticker}")
    
    try:
        prompt = f"""Provide brief general information about the company with 
ticker symbol {ticker}. Include what the company does and its industry.
Note: This is general knowledge and does not include current stock prices."""
        
        response = llm.invoke(prompt)
        result = f"{response.content}\n\nSource: LLM general knowledge (not current data)"
        
        return (True, result)
        
    except Exception as e:
        return (False, f"LLM error: {str(e)}")


print("Data source functions defined!")
print("\nAvailable functions:")
print("  - fetch_from_stock_api(ticker) -> (success, result)")
print("  - fetch_from_web_search(ticker) -> (success, result)")
print("  - fetch_from_llm_knowledge(ticker) -> (success, result)")

## Part 1: Implement Workflow Nodes

Each node calls its corresponding data source function and updates the state based on success or failure.

**Key behaviors:**
- On success: Set `result` and appropriate `data_quality`
- On failure: Append error to `error_log`, increment `degradation_level`, leave `result` as None

In [ ]:
# Provided: initialize_node

def initialize_node(state: StockWorkflowState) -> StockWorkflowState:
    """Initialize state and log the start of the workflow."""
    logger.info(f"=== Starting stock query for: {state['ticker']} ===")
    return {
        "ticker": state["ticker"],
        "degradation_level": 0,
        "result": None,
        "error_log": [],
        "data_quality": "high"
    }


print("initialize_node defined!")

In [ ]:
# TODO: Implement attempt_api_node

def attempt_api_node(state: StockWorkflowState) -> StockWorkflowState:
    """
    Level 0 node: Attempt to fetch from stock API.
    
    Steps:
    1. Call fetch_from_stock_api(state["ticker"])
    2. If success:
       - Set result to the response
       - Set data_quality to "high"
       - Keep degradation_level at 0
    3. If failure:
       - Leave result as None
       - Append error message to error_log
       - Set degradation_level to 1
    
    Returns:
        Updated StockWorkflowState
    """
    # TODO: Implement the logic
    pass


print("attempt_api_node defined!")

In [ ]:
# TODO: Implement attempt_search_node

def attempt_search_node(state: StockWorkflowState) -> StockWorkflowState:
    """
    Level 1 node: Attempt to fetch from web search.
    
    Steps:
    1. Call fetch_from_web_search(state["ticker"])
    2. If success:
       - Set result to the response
       - Set data_quality to "medium"
       - Keep degradation_level at 1
    3. If failure:
       - Leave result as None
       - Append error message to error_log
       - Set degradation_level to 2
    
    Returns:
        Updated StockWorkflowState
    """
    # TODO: Implement the logic
    pass


print("attempt_search_node defined!")

In [ ]:
# TODO: Implement attempt_llm_node

def attempt_llm_node(state: StockWorkflowState) -> StockWorkflowState:
    """
    Level 2 node: Fetch from LLM knowledge (last resort).
    
    Steps:
    1. Call fetch_from_llm_knowledge(state["ticker"])
    2. If success:
       - Set result to the response
       - Set data_quality to "low"
    3. If failure (rare):
       - Set result to "Unable to retrieve any information"
       - Set data_quality to "none"
    
    Returns:
        Updated StockWorkflowState
    """
    # TODO: Implement the logic
    pass


print("attempt_llm_node defined!")

In [ ]:
# Provided: format_output_node

def format_output_node(state: StockWorkflowState) -> StockWorkflowState:
    """Format the final output with quality indicator."""
    quality_badges = {
        "high": "[HIGH QUALITY] Real-time API data",
        "medium": "[MEDIUM QUALITY] Web search data",
        "low": "[LOW QUALITY] General LLM knowledge",
        "none": "[NO DATA] All sources failed"
    }
    
    badge = quality_badges.get(state["data_quality"], "Unknown")
    formatted = f"{state['result']}\n\n{badge}"
    
    if state["error_log"]:
        formatted += f"\n\nDegraded from level 0 due to {len(state['error_log'])} error(s)"
    
    logger.info(f"=== Query completed at level {state['degradation_level']} ===")
    
    return {
        "ticker": state["ticker"],
        "degradation_level": state["degradation_level"],
        "result": formatted,
        "error_log": state["error_log"],
        "data_quality": state["data_quality"]
    }


print("format_output_node defined!")

## Part 2: Implement Routing Functions

The routing functions examine the state and decide where to go next:
- If `result` is not None (success) -> route to `format_output`
- If `result` is None (failure) -> route to next fallback level

In [ ]:
# TODO: Implement route_after_api

def route_after_api(state: StockWorkflowState) -> Literal["format_output", "attempt_search"]:
    """
    Route after API attempt.
    
    Check if result is not None:
        - If result exists: return "format_output"
        - If result is None: return "attempt_search"
    
    Returns:
        The name of the next node to execute
    """
    # TODO: Implement the routing logic
    pass


print("route_after_api defined!")

In [ ]:
# TODO: Implement route_after_search

def route_after_search(state: StockWorkflowState) -> Literal["format_output", "attempt_llm"]:
    """
    Route after search attempt.
    
    Check if result is not None:
        - If result exists: return "format_output"
        - If result is None: return "attempt_llm"
    
    Returns:
        The name of the next node to execute
    """
    # TODO: Implement the routing logic
    pass


print("route_after_search defined!")

## Part 3: Assemble the Workflow

Connect all the nodes and routing functions into a complete LangGraph workflow.

```
START -> initialize -> attempt_api
                          |
            [success] -> format_output -> END
            [failure] -> attempt_search
                              |
                [success] -> format_output -> END
                [failure] -> attempt_llm -> format_output -> END
```

In [ ]:
# Build the workflow graph
stock_workflow = StateGraph(StockWorkflowState)

# Add nodes
stock_workflow.add_node("initialize", initialize_node)
stock_workflow.add_node("attempt_api", attempt_api_node)
stock_workflow.add_node("attempt_search", attempt_search_node)
stock_workflow.add_node("attempt_llm", attempt_llm_node)
stock_workflow.add_node("format_output", format_output_node)

# Add edges - connect START to initialize, then to attempt_api
stock_workflow.add_edge(START, "initialize")
stock_workflow.add_edge("initialize", "attempt_api")

# TODO: Add conditional edge from attempt_api using route_after_api
# Use add_conditional_edges with mapping: {"format_output": "format_output", "attempt_search": "attempt_search"}


# TODO: Add conditional edge from attempt_search using route_after_search
# Use add_conditional_edges with mapping: {"format_output": "format_output", "attempt_llm": "attempt_llm"}


# LLM always goes to format_output (last resort)
stock_workflow.add_edge("attempt_llm", "format_output")

# format_output goes to END
stock_workflow.add_edge("format_output", END)

# Compile the workflow
stock_app = stock_workflow.compile()

print("Workflow compiled!")

## Run Your Implementation

Test your graceful degradation workflow with different scenarios.

In [ ]:
# Test 1: Normal operation (API succeeds)
print("=" * 60)
print("TEST 1: Normal Operation - API Success")
print("=" * 60)

SIMULATE_API_FAILURE = False
SIMULATE_SEARCH_FAILURE = False

result = stock_app.invoke({
    "ticker": "AAPL",
    "degradation_level": 0,
    "result": None,
    "error_log": [],
    "data_quality": "high"
})

print(f"\nFinal Result:\n{result['result']}")
print(f"\nDegradation Level: {result['degradation_level']}")
print(f"Errors: {result['error_log']}")

In [ ]:
# Test 2: API fails, search succeeds
print("=" * 60)
print("TEST 2: API Failure - Search Fallback")
print("=" * 60)

SIMULATE_API_FAILURE = True
SIMULATE_SEARCH_FAILURE = False

result = stock_app.invoke({
    "ticker": "GOOGL",
    "degradation_level": 0,
    "result": None,
    "error_log": [],
    "data_quality": "high"
})

print(f"\nFinal Result:\n{result['result']}")
print(f"\nDegradation Level: {result['degradation_level']}")
print(f"Errors: {result['error_log']}")

In [ ]:
# Test 3: Both API and search fail, LLM fallback
print("=" * 60)
print("TEST 3: Double Failure - LLM Fallback")
print("=" * 60)

SIMULATE_API_FAILURE = True
SIMULATE_SEARCH_FAILURE = True

result = stock_app.invoke({
    "ticker": "MSFT",
    "degradation_level": 0,
    "result": None,
    "error_log": [],
    "data_quality": "high"
})

print(f"\nFinal Result:\n{result['result']}")
print(f"\nDegradation Level: {result['degradation_level']}")
print(f"Errors: {result['error_log']}")

# Reset flags
SIMULATE_API_FAILURE = False
SIMULATE_SEARCH_FAILURE = False